# Construcción de df_pacientes_trayectorias
Este notebook construye la base `df_pacientes_trayectorias` a partir de `df_base_limpia` (episodios) y `df_traslados_strict` (transiciones).
Cada fila representará **un paciente con su trayectoria hospitalaria**.

Se implementarán y compararán dos metodologías:
1. **Metodología 1 — Anclada en Traslados (Principal)**
2. **Metodología 2 — Desde Episodios (Alternativa)**


In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.config import *

# Carga de datos
df_base = pd.read_excel("../data/final_data/df_base_limpia.xlsx")
# df_traslados = pd.read_excel("../data/final_data/df_traslados_strict.xlsx")

# Ordenar
df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'])
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'])
df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso']).copy()

# df_traslados['fecha_egreso_origen'] = pd.to_datetime(df_traslados['fecha_egreso_origen'])
# df_traslados['fecha_ingreso_destino'] = pd.to_datetime(df_traslados['fecha_ingreso_destino'])


## 1. Limpieza (nivel paciente — filtros estrictos)
Aplicamos filtros a `df_base` para obtener una lista de `pacientes_validos`.

In [2]:
# --- fechas incompletas
df_base['flag_fechas_incompletas'] = df_base['fecha_ingreso'].isna() | df_base['fecha_egreso'].isna()

# --- consistencia temporal (tolerancia de 10 mins)
TOL_MINUTOS = 10
df_base['delta_min'] = (df_base['fecha_egreso'] - df_base['fecha_ingreso']).dt.total_seconds() / 60
df_base['flag_fechas_invalidas'] = df_base['delta_min'] < -TOL_MINUTOS

# --- consistencia de edad (delta > 2)
df_base['edad_next'] = df_base.groupby('paciente_id')['edad'].shift(-1)
df_base['delta_edad_episodios'] = df_base['edad_next'] - df_base['edad']
df_base['flag_edad_inconsistente'] = df_base['delta_edad_episodios'].abs() > 2

# --- egreso administrativo final
idx_last = df_base.groupby('paciente_id')['fecha_ingreso'].idxmax()
df_last = df_base.loc[idx_last, ['paciente_id', 'tipo_egreso']].copy()
df_last['flag_fin_admin'] = df_last['tipo_egreso'] == 'administrativo'

# --- agregación por paciente
df_flags_paciente = df_base.groupby('paciente_id').agg({
    'flag_fechas_incompletas': 'max',
    'flag_fechas_invalidas': 'max',
    'flag_edad_inconsistente': 'max'
}).reset_index()

df_flags_paciente = df_flags_paciente.merge(df_last[['paciente_id', 'flag_fin_admin']], on='paciente_id', how='left')

# paciente válido si no incumple ninguna regla
df_flags_paciente['paciente_valido'] = (
    ~df_flags_paciente['flag_fechas_incompletas'] &
    ~df_flags_paciente['flag_fechas_invalidas'] &
    ~df_flags_paciente['flag_edad_inconsistente'] &
    ~df_flags_paciente['flag_fin_admin']
)

pacientes_validos = df_flags_paciente.loc[df_flags_paciente['paciente_valido'], 'paciente_id']
df_base_filtrada = df_base[df_base['paciente_id'].isin(pacientes_validos)].copy()

print(f"Total pacientes originales: {df_base['paciente_id'].nunique()}")
print(f"Total pacientes válidos: {len(pacientes_validos)}")


Total pacientes originales: 22729
Total pacientes válidos: 19980


# METODOLOGÍA sandwich

In [3]:
# METODOLOGÍA PRINCIPAL — LÓGICA SÁNDWICH (Desde df_base_filtrada)
# =============================================================================

def construir_trayectoria_sandwich(grp):
    # PAN DE ARRIBA: Ingreso
    fecha_ingreso_paciente = grp['fecha_ingreso'].min()
    
    # RELLENO VS PAN DE ABAJO
    mask_traslado = grp['tipo_egreso'].astype(str).str.lower().str.contains('traslado')
    df_traslados = grp[mask_traslado].sort_values('fecha_ingreso')
    df_finales = grp[~mask_traslado].sort_values('fecha_ingreso')
    
    flags = {'sin_evento_final': 0, 'multiples_finales': 0}
    
    # Definir desenlace y egreso (PAN DE ABAJO)
    if len(df_finales) == 0:
        flags['sin_evento_final'] = 1
        fecha_egreso_paciente = grp['fecha_egreso'].max()
        desenlace = 'solo_traslados'
    else:
        if len(df_finales) > 1:
            flags['multiples_finales'] = 1
        fecha_egreso_paciente = df_finales.iloc[-1]['fecha_egreso']
        desenlace = df_finales.iloc[-1]['tipo_egreso']

    # ARMAR EL SÁNDWICH: Primero los traslados, último el final
    hospitales_seq = df_traslados['hospital_origen'].tolist() + df_finales['hospital_origen'].tolist()
    
    # Limpiar repetidos consecutivos
    hospitales_limpios = []
    if hospitales_seq:
        hospitales_limpios = [hospitales_seq[0]]
        for h in hospitales_seq[1:]:
            if h != hospitales_limpios[-1]:
                hospitales_limpios.append(h)
                
    # También guardamos la lista real (no solo string) para facilitar el paso 3
    return pd.Series({
        'lista_hospitales': hospitales_limpios, # Lista real (útil para derivar traslados)
        'trayectoria_hospitalaria': str(hospitales_limpios), # String para Excel/visualización
        'hospital_inicio': hospitales_limpios[0] if hospitales_limpios else None,
        'hospital_final': hospitales_limpios[-1] if hospitales_limpios else None,
        'fecha_ingreso_paciente': fecha_ingreso_paciente,
        'fecha_egreso_paciente': fecha_egreso_paciente,
        'desenlace': desenlace,
        'flag_sin_evento_final': flags['sin_evento_final'],
        'flag_multiples_finales': flags['multiples_finales']
    })

# Ejecutamos la creación de trayectorias
df_trayectorias_limpias = df_base_filtrada.groupby('paciente_id').apply(construir_trayectoria_sandwich).reset_index()

In [4]:
# DERIVACIÓN DE TRASLADOS DESDE LA TRAYECTORIA
# =============================================================================
registros_traslados = []

for _, row in df_trayectorias_limpias.iterrows():
    paciente = row['paciente_id']
    hospitales = row['lista_hospitales']
    
    # Si pasó por más de un hospital, extraemos los pares Origen -> Destino
    if len(hospitales) > 1:
        for i in range(len(hospitales) - 1):
            registros_traslados.append({
                'paciente_id': paciente,
                'hospital_origen': hospitales[i],
                'hospital_destino': hospitales[i+1],
                'paso_trayectoria': i + 1 # Nos dice si es su 1er traslado, 2do, etc.
            })

# Construimos el nuevo dataframe de traslados oficial
df_traslados_derivados = pd.DataFrame(registros_traslados)

# Limpiamos la columna 'lista_hospitales' (que era temporal) antes de exportar
df_trayectorias_final = df_trayectorias_limpias.drop(columns=['lista_hospitales'])

# Guardamos los resultados
df_trayectorias_final.to_excel("../data/final_data/df_pacientes_trayectorias_oficial.xlsx", index=False)
df_traslados_derivados.to_excel("../data/final_data/df_traslados_derivados.xlsx", index=False)

print(f"Trayectorias construidas: {len(df_trayectorias_final)}")
print(f"Traslados reales derivados: {len(df_traslados_derivados)}")

Trayectorias construidas: 19980
Traslados reales derivados: 2049


In [ ]:
# 3. Trayectorias de un solo hospital
df_triviales = df_base_filtrada[df_base_filtrada['paciente_id'].isin(pacientes_sin_traslado)].copy()

def trayectoria_trivial(grp):
    h_seq = grp.sort_values('fecha_ingreso')['hospital_origen'].tolist()
    # colapsar repetidos
    hospitales_limpios = [h_seq[0]]
    for h in h_seq[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)
            
    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final': hospitales_limpios[-1],
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        'flag_salto_inconsistente': 0,
        'flag_loop': 0,
        'flag_duplicado_consecutivo': 0
    })

if len(df_triviales) > 0:
    df_tray_triviales = df_triviales.groupby('paciente_id').apply(trayectoria_trivial).reset_index()
else:
    df_tray_triviales = pd.DataFrame()

# 4. Unificación
df_tray_v1 = pd.concat([df_tray_conectadas, df_tray_triviales], ignore_index=True)


In [ ]:
# 5. Enriquecimiento y 6. Desenlace
df_metrics = df_base_filtrada.groupby('paciente_id').agg({
    'fecha_ingreso': 'min',
    'fecha_egreso': 'max',
    'hospital_origen': 'count'
}).rename(columns={
    'fecha_ingreso': 'fecha_primer_ingreso',
    'fecha_egreso': 'fecha_ultimo_egreso',
    'hospital_origen': 'n_episodios'
}).reset_index()

df_metrics['duracion_total'] = (df_metrics['fecha_ultimo_egreso'] - df_metrics['fecha_primer_ingreso']).dt.days

def calcular_desenlace(grp):
    grp = grp.sort_values('fecha_ingreso')
    muerte = grp[grp['tipo_egreso'] == 'muerte']
    
    if len(muerte) > 0:
        return pd.Series({'desenlace': 'muerte'})
    
    ultimo = grp.iloc[-1]['tipo_egreso']
    if pd.isna(ultimo) or ultimo == '':
        return pd.Series({'desenlace': 'desconocido'})
    
    return pd.Series({'desenlace': ultimo})

df_desenlace = df_base_filtrada.groupby('paciente_id').apply(calcular_desenlace).reset_index()

# Merge Final M1
df_pacientes_trayectorias_v1 = df_tray_v1.merge(df_metrics, on='paciente_id', how='left').merge(df_desenlace, on='paciente_id', how='left')
df_pacientes_trayectorias_v1.to_excel("../data/final_data/df_pacientes_trayectorias_v1.xlsx", index=False)


# METODOLOGÍA 2 — DESDE EPISODIOS (ALTERNATIVA)

In [ ]:
# 2. Construcción directa desde df_base_limpia
def construir_trayectoria_m2(grp):
    grp = grp.sort_values('fecha_ingreso')
    
    hospitales = grp['hospital_origen'].tolist()
    fechas_in = grp['fecha_ingreso'].tolist()
    fechas_out = grp['fecha_egreso'].tolist()
    
    flags = {
        'overlap': 0,
        'gap_grande': 0
    }
    
    hospitales_limpios = [hospitales[0]]
    
    for i in range(1, len(grp)):
        prev_out = fechas_out[i-1]
        curr_in = fechas_in[i]
        
        delta_dias = (curr_in - prev_out).total_seconds() / (3600*24)
        
        if delta_dias < 0:
            flags['overlap'] = 1
        elif delta_dias > 5:
            flags['gap_grande'] = 1
            
        if hospitales[i] != hospitales_limpios[-1]:
            hospitales_limpios.append(hospitales[i])
            
    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final': hospitales_limpios[-1],
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        'flag_overlap': flags['overlap'],
        'flag_gap_grande': flags['gap_grande']
    })

df_tray_v2 = df_base_filtrada.groupby('paciente_id').apply(construir_trayectoria_m2).reset_index()

df_pacientes_trayectorias_v2 = df_tray_v2.merge(df_metrics, on='paciente_id', how='left').merge(df_desenlace, on='paciente_id', how='left')
df_pacientes_trayectorias_v2.to_excel("../data/final_data/df_pacientes_trayectorias_v2.xlsx", index=False)


# ⚖️ COMPARACIÓN ENTRE METODOLOGÍAS

In [ ]:
# 1. Cobertura y 2. Trayectorias
print("=== METODOLOGÍA 1 (Anclada en Traslados) ===")
print(f"Pacientes: {len(df_pacientes_trayectorias_v1)}")
print(f"Hospitales únicos promedio: {df_pacientes_trayectorias_v1['n_hospitales_unicos'].mean():.2f}")
print(f"Deselance muerte: {(df_pacientes_trayectorias_v1['desenlace'] == 'muerte').sum()}")

print("\n=== METODOLOGÍA 2 (Desde Episodios) ===")
print(f"Pacientes: {len(df_pacientes_trayectorias_v2)}")
print(f"Hospitales únicos promedio: {df_pacientes_trayectorias_v2['n_hospitales_unicos'].mean():.2f}")
print(f"Deselance muerte: {(df_pacientes_trayectorias_v2['desenlace'] == 'muerte').sum()}")

# 3. Diferencias estructurales
df_comp = df_pacientes_trayectorias_v1[['paciente_id', 'trayectoria_hospitalaria', 'hospital_final', 'desenlace']].merge(
    df_pacientes_trayectorias_v2[['paciente_id', 'trayectoria_hospitalaria', 'hospital_final', 'desenlace']],
    on='paciente_id', suffixes=('_v1', '_v2')
)

df_comp['diff_trayectoria'] = df_comp['trayectoria_hospitalaria_v1'] != df_comp['trayectoria_hospitalaria_v2']
df_comp['diff_hospital_final'] = df_comp['hospital_final_v1'] != df_comp['hospital_final_v2']
df_comp['diff_desenlace'] = df_comp['desenlace_v1'] != df_comp['desenlace_v2']

print("\n=== DIFERENCIAS ESTRUCTURALES ===")
print(f"Diferencia en trayectoria: {df_comp['diff_trayectoria'].sum()} pacientes")
print(f"Diferencia en hospital final: {df_comp['diff_hospital_final'].sum()} pacientes")
print(f"Diferencia en desenlace: {df_comp['diff_desenlace'].sum()} pacientes")


In [ ]:
# 4. Casos conflictivos (MUY IMPORTANTE)
conflictos = df_comp[df_comp['diff_trayectoria']].head()

print("Ejemplos de diferencias en trayectorias:")
for _, row in conflictos.iterrows():
    print(f"\nPaciente: {row['paciente_id']}")
    print(f"V1: {row['trayectoria_hospitalaria_v1']}")
    print(f"V2: {row['trayectoria_hospitalaria_v2']}")


## 5. Conclusión esperada

**¿Qué metodología representa mejor la red hospitalaria?**
La **Metodología 1 (Anclada en Traslados)** representa mejor el verdadero flujo logístico de la red. Al depender explícitamente de los "pares" de episodios validados y verificados temporalmente (y filtrados por saltos/overlaps en `22.TRASLADOS.ipynb`), evitamos conectar internaciones aisladas en el tiempo que un paciente pueda tener a lo largo de los años por motivos no relacionados. 

**¿Cuál es más robusta a errores administrativos?**
La **Metodología 1**. Al usar `df_traslados_strict`, que exige un "motivo de traslado" validado por el sistema o por tiempos logísticos consistentes, mitigamos el ruido introducido por altas mal cargadas o episodios fragmentados dentro del mismo hospital o red. La Metodología 2 es susceptible a falsas derivaciones: si un paciente recibe un alta y luego de 4 meses vuelve a otro hospital, la V2 lo considera una trayectoria conectada que fluye, lo cual es incorrecto a nivel operativo de derivaciones críticas.

**¿Qué sesgos introduce cada una?**
- **Metodología 1 (Sesgo de omisión):** Puede perder "traslados reales" si el origen no documentó bien la salida o si hay un gap administrativo justo mayor a la tolerancia, cayendo en pacientes con trayectorias triviales cuando en realidad fluyeron.
- **Metodología 2 (Sesgo de falsa conectividad):** Tiende a sobre-conectar episodios. Una persona con enfermedades crónicas puede asistir a múltiples hospitales durante 3 años, y la Metodología 2 creará una "super-trayectoria" larga que parece una derivación secuencial pero que en realidad son episodios independientes.


---

# 🔧 METODOLOGÍA 1 — AJUSTADA (v1_ajustada)

Se incorporan dos correcciones al modelo v1 basadas en patrones observados en los datos:

**Problema 1 — Duplicados con traslados:**  
Muchos pacientes tienen varios episodios con `tipo_egreso == 'traslado'`
pero un solo episodio con `alta` o `muerte`. 
Ese episodio no-traslado es el **cierre clínico real**, 
pero a veces las fechas no lo ordenan correctamente.

**Problema 2 — Definición de ingreso/egreso:**  
- `fecha_ingreso_paciente = min(fecha_ingreso)` (siempre)
- `fecha_egreso_paciente = fecha_egreso` del episodio cuyo `tipo_egreso != 'traslado'`

**Estrategia:** prioridad lógica > fechas.


In [ ]:
# ==============================================================================
# AJUSTE 1 — CLASIFICACIÓN DE EPISODIOS POR ROL (intermedios vs final)
# ==============================================================================

# Marcar qué episodios son traslados (intermedios) vs finales clínicos
df_base_filtrada = df_base_filtrada.copy()

df_base_filtrada['es_traslado'] = df_base_filtrada['tipo_egreso'] == 'traslado'

# Por paciente: ¿tiene al menos un episodio NO traslado?
tiene_final = (
    df_base_filtrada
    .groupby('paciente_id')['es_traslado']
    .apply(lambda x: (x == False).any())
    .rename('tiene_final_clinico')
    .reset_index()
)

df_base_filtrada = df_base_filtrada.merge(tiene_final, on='paciente_id', how='left')

print("=== DISTRIBUCIÓN DE TIPOS DE EPISODIO ===")
print(df_base_filtrada['tipo_egreso'].value_counts())
print(f"\nPacientes CON episodio final clínico:    {tiene_final['tiene_final_clinico'].sum()}")
print(f"Pacientes SIN episodio final clínico:     {(~tiene_final['tiene_final_clinico']).sum()}")
print(f"  -> estos quedan flagueados, NO eliminados")


In [ ]:
# ==============================================================================
# AJUSTE 2 — FECHA DE INGRESO Y EGRESO REAL POR PACIENTE
# ==============================================================================

def calcular_ingreso_egreso_ajustado(grp):
    """
    Ingreso real  = min(fecha_ingreso)  [siempre, no depende del motivo]
    Egreso real   = fecha_egreso del episodio con tipo_egreso != 'traslado'
                    Si no existe, usar max(fecha_egreso) y marcar flag.
    """
    grp = grp.sort_values('fecha_ingreso')

    # Ingreso
    fecha_ingreso_paciente = grp['fecha_ingreso'].min()

    # Episodios finales (no traslado)
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        # Todos son traslado → inconsistente
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': grp['fecha_egreso'].max(),
            'flag_sin_final_clinico': True,
            'flag_multiples_finales': False,
            'tipo_egreso_final': 'desconocido'
        })
    elif len(finales) == 1:
        ep_final = finales.iloc[0]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': False,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })
    else:
        # Múltiples candidatos finales → usar el de mayor fecha y flaggear
        ep_final = finales.sort_values('fecha_egreso').iloc[-1]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': True,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })

df_ingreso_egreso = (
    df_base_filtrada
    .groupby('paciente_id')
    .apply(calcular_ingreso_egreso_ajustado)
    .reset_index()
)

print("=== INGRESO/EGRESO AJUSTADO ===")
print(f"Pacientes sin final clínico (flag):   {df_ingreso_egreso['flag_sin_final_clinico'].sum()}")
print(f"Pacientes con múltiples finales (flag): {df_ingreso_egreso['flag_multiples_finales'].sum()}")
print("\nDistribución tipo_egreso_final:")
print(df_ingreso_egreso['tipo_egreso_final'].value_counts())


In [ ]:
# ==============================================================================
# AJUSTE 3 — CONSTRUCCIÓN DE TRAYECTORIA CON NODO FINAL ORDENADO POR LÓGICA
# ==============================================================================

def construir_trayectoria_v1_ajustada(grp):
    """
    Construcción de trayectoria con prioridad lógica sobre fechas:
    
    1. Separa los traslados del df_traslados_strict (intermedios)
       y el episodio final (no traslado) desde df_base_filtrada.
    2. Ordena los traslados por fecha_egreso_origen.
    3. Ancla el episodio final al último lugar, independientemente
       de la fecha (prioridad lógica > orden temporal).
    4. Levanta flags en casos ambiguos.
    """
    grp = grp.sort_values('fecha_egreso_origen')

    flags = {
        'salto_inconsistente': 0,
        'loop': 0,
        'duplicado_consecutivo': 0
    }

    current_hospital = grp.iloc[0]['hospital_origen']
    hospitales = [current_hospital]

    for _, row in grp.iterrows():
        origen = row['hospital_origen']
        destino = row['hospital_destino']

        if origen != current_hospital:
            flags['salto_inconsistente'] = 1

        if destino == current_hospital:
            flags['duplicado_consecutivo'] = 1
        elif destino in hospitales:
            flags['loop'] = 1

        hospitales.append(destino)
        current_hospital = destino

    # Colapsar duplicados consecutivos
    hospitales_limpios = [hospitales[0]]
    for h in hospitales[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final_traslados': hospitales_limpios[-1],
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        **{f'flag_{k}': v for k, v in flags.items()}
    })


# Aplicar a pacientes con traslados
df_tray_ajustada_conectada = (
    df_traslados_filtrado
    .groupby('paciente_id')
    .apply(construir_trayectoria_v1_ajustada)
    .reset_index()
)

# Ahora — AJUSTE CLAVE: incorporar el nodo final desde df_base_filtrada
# El hospital del episodio final real (no-traslado) es el verdadero último nodo
df_nodo_final = (
    df_base_filtrada[df_base_filtrada['tipo_egreso'] != 'traslado']
    .sort_values(['paciente_id', 'fecha_egreso'])
    .groupby('paciente_id')
    .last()[['hospital_origen', 'tipo_egreso']]
    .rename(columns={'hospital_origen': 'hospital_nodo_final', 'tipo_egreso': 'tipo_egreso_final_ep'})
    .reset_index()
)

# Unir con trayectorias conectadas
df_tray_ajustada_conectada = df_tray_ajustada_conectada.merge(
    df_nodo_final, on='paciente_id', how='left'
)

# Flag: ¿el nodo final del traslado difiere del nodo final clínico?
df_tray_ajustada_conectada['flag_nodo_final_discrepante'] = (
    df_tray_ajustada_conectada['hospital_final_traslados'] !=
    df_tray_ajustada_conectada['hospital_nodo_final']
)

# Hospital final ajustado = hospital del episodio clínico final
# Si no existe (todos traslados), mantener el del último traslado
df_tray_ajustada_conectada['hospital_final'] = (
    df_tray_ajustada_conectada['hospital_nodo_final']
    .fillna(df_tray_ajustada_conectada['hospital_final_traslados'])
)

print("=== NODO FINAL DISCREPANTE (traslados vs clínico) ===")
print(df_tray_ajustada_conectada['flag_nodo_final_discrepante'].value_counts())


In [ ]:
# ==============================================================================
# TRAYECTORIAS TRIVIALES AJUSTADAS (sin traslados en df_traslados_strict)
# ==============================================================================

df_triviales_aj = df_base_filtrada[
    df_base_filtrada['paciente_id'].isin(pacientes_sin_traslado)
].copy()

def trayectoria_trivial_ajustada(grp):
    grp = grp.sort_values('fecha_ingreso')

    # Separar traslados de no-traslados
    intermedios = grp[grp['tipo_egreso'] == 'traslado']['hospital_origen'].tolist()
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        hospital_final = grp['hospital_origen'].iloc[-1]
        flag_sin_final = True
    else:
        # El hospital final es el del primer episodio no-traslado (más clínico)
        hospital_final = finales.sort_values('fecha_egreso').iloc[-1]['hospital_origen']
        flag_sin_final = False

    # Secuencia: intermedios + final
    h_seq = grp['hospital_origen'].tolist()
    hospitales_limpios = [h_seq[0]]
    for h in h_seq[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final': hospital_final,
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        'flag_salto_inconsistente': 0,
        'flag_loop': 0,
        'flag_duplicado_consecutivo': 0,
        'flag_nodo_final_discrepante': hospital_final != hospitales_limpios[-1],
        'flag_sin_final_clinico': flag_sin_final
    })

if len(df_triviales_aj) > 0:
    df_tray_triviales_aj = (
        df_triviales_aj
        .groupby('paciente_id')
        .apply(trayectoria_trivial_ajustada)
        .reset_index()
    )
else:
    df_tray_triviales_aj = pd.DataFrame()


In [ ]:
# ==============================================================================
# UNIFICACIÓN Y MERGE FINAL → df_pacientes_trayectorias_v1_ajustada
# ==============================================================================

df_tray_ajustada = pd.concat([
    df_tray_ajustada_conectada,
    df_tray_triviales_aj
], ignore_index=True)

# Merge con métricas ajustadas de ingreso/egreso
df_tray_ajustada_final = (
    df_tray_ajustada
    .merge(df_ingreso_egreso, on='paciente_id', how='left')
    .merge(
        df_base_filtrada.groupby('paciente_id')['hospital_origen']
        .count().reset_index().rename(columns={'hospital_origen': 'n_episodios'}),
        on='paciente_id', how='left'
    )
)

# Desenlace desde tipo_egreso_final (ya calculado con la lógica ajustada)
df_tray_ajustada_final['desenlace'] = df_tray_ajustada_final['tipo_egreso_final']

# Métricas de duración usando ingreso/egreso ajustado
df_tray_ajustada_final['duracion_total'] = (
    (df_tray_ajustada_final['fecha_egreso_paciente'] - df_tray_ajustada_final['fecha_ingreso_paciente'])
    .dt.days
)

# Guardar
df_tray_ajustada_final.to_excel("../data/final_data/df_pacientes_trayectorias_v1_ajustada.xlsx", index=False)

print("=== RESUMEN FINAL — V1 AJUSTADA ===")
print(f"Pacientes totales:                {len(df_tray_ajustada_final)}")
print(f"Con nodo final discrepante:       {df_tray_ajustada_final['flag_nodo_final_discrepante'].sum()}")
print(f"Sin final clínico (flag):         {df_tray_ajustada_final['flag_sin_final_clinico'].sum()}")
print(f"Con múltiples finales (flag):     {df_tray_ajustada_final.get('flag_multiples_finales', pd.Series(False)).sum()}")
print(f"\nDesenlace:")
print(df_tray_ajustada_final['desenlace'].value_counts())
print(f"\nHospitales únicos promedio:       {df_tray_ajustada_final['n_hospitales_unicos'].mean():.2f}")
print(f"Duración total promedio (días):   {df_tray_ajustada_final['duracion_total'].median():.1f} (mediana)")


In [ ]:
# ==============================================================================
# VALIDACIÓN — COMPARAR V1 original vs V1 AJUSTADA
# ==============================================================================

df_v1_orig = pd.read_excel("../data/final_data/df_pacientes_trayectorias_v1.xlsx")

comp = df_v1_orig[['paciente_id', 'hospital_final', 'desenlace']].merge(
    df_tray_ajustada_final[['paciente_id', 'hospital_final', 'desenlace', 'flag_nodo_final_discrepante']],
    on='paciente_id', suffixes=('_orig', '_aj'), how='inner'
)

comp['cambio_hospital_final'] = comp['hospital_final_orig'] != comp['hospital_final_aj']
comp['cambio_desenlace']      = comp['desenlace_orig'] != comp['desenlace_aj']

print("=== V1 ORIGINAL vs V1 AJUSTADA ===")
print(f"Pacientes comparados:              {len(comp)}")
print(f"Con cambio en hospital final:      {comp['cambio_hospital_final'].sum()} ({comp['cambio_hospital_final'].mean():.1%})")
print(f"Con cambio en desenlace:           {comp['cambio_desenlace'].sum()} ({comp['cambio_desenlace'].mean():.1%})")

# Muestra de casos que cambiaron
print("\n=== SAMPLE DE CASOS CON NODO FINAL DISTINTO ===")
sample_cambios = comp[comp['cambio_hospital_final']].sample(min(5, comp['cambio_hospital_final'].sum()), random_state=42)
for _, row in sample_cambios.iterrows():
    ep_cols = ['fecha_ingreso', 'fecha_egreso', 'hospital_origen', 'tipo_egreso']
    print(f"\nPaciente {row['paciente_id']}")
    print(f"  Hospital final ORIGINAL: {row['hospital_final_orig']}")
    print(f"  Hospital final AJUSTADO: {row['hospital_final_aj']}")
    from IPython.display import display
    display(df_base_filtrada[df_base_filtrada['paciente_id'] == row['paciente_id']][ep_cols])


---

# 🔧 METODOLOGÍA 1 — AJUSTADA (v1_ajustada)

Se incorporan dos correcciones al modelo v1 basadas en patrones observados en los datos:

**Problema 1 — Duplicados con traslados:**  
Muchos pacientes tienen varios episodios con `tipo_egreso == 'traslado'`
pero un solo episodio con `alta` o `muerte`. 
Ese episodio no-traslado es el **cierre clínico real**, 
pero a veces las fechas no lo ordenan correctamente.

**Problema 2 — Definición de ingreso/egreso:**  
- `fecha_ingreso_paciente = min(fecha_ingreso)` (siempre)
- `fecha_egreso_paciente = fecha_egreso` del episodio cuyo `tipo_egreso != 'traslado'`

**Estrategia:** prioridad lógica > fechas.


In [ ]:
# ==============================================================================
# AJUSTE 1 — CLASIFICACIÓN DE EPISODIOS POR ROL (intermedios vs final)
# ==============================================================================

# Marcar qué episodios son traslados (intermedios) vs finales clínicos
df_base_filtrada = df_base_filtrada.copy()

df_base_filtrada['es_traslado'] = df_base_filtrada['tipo_egreso'] == 'traslado'

# Por paciente: ¿tiene al menos un episodio NO traslado?
tiene_final = (
    df_base_filtrada
    .groupby('paciente_id')['es_traslado']
    .apply(lambda x: (x == False).any())
    .rename('tiene_final_clinico')
    .reset_index()
)

df_base_filtrada = df_base_filtrada.merge(tiene_final, on='paciente_id', how='left')

print("=== DISTRIBUCIÓN DE TIPOS DE EPISODIO ===")
print(df_base_filtrada['tipo_egreso'].value_counts())
print(f"\nPacientes CON episodio final clínico:    {tiene_final['tiene_final_clinico'].sum()}")
print(f"Pacientes SIN episodio final clínico:     {(~tiene_final['tiene_final_clinico']).sum()}")
print(f"  -> estos quedan flagueados, NO eliminados")


In [ ]:
# ==============================================================================
# AJUSTE 2 — FECHA DE INGRESO Y EGRESO REAL POR PACIENTE
# ==============================================================================

def calcular_ingreso_egreso_ajustado(grp):
    """
    Ingreso real  = min(fecha_ingreso)  [siempre, no depende del motivo]
    Egreso real   = fecha_egreso del episodio con tipo_egreso != 'traslado'
                    Si no existe, usar max(fecha_egreso) y marcar flag.
    """
    grp = grp.sort_values('fecha_ingreso')

    # Ingreso
    fecha_ingreso_paciente = grp['fecha_ingreso'].min()

    # Episodios finales (no traslado)
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        # Todos son traslado → inconsistente
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': grp['fecha_egreso'].max(),
            'flag_sin_final_clinico': True,
            'flag_multiples_finales': False,
            'tipo_egreso_final': 'desconocido'
        })
    elif len(finales) == 1:
        ep_final = finales.iloc[0]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': False,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })
    else:
        # Múltiples candidatos finales → usar el de mayor fecha y flaggear
        ep_final = finales.sort_values('fecha_egreso').iloc[-1]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': True,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })

df_ingreso_egreso = (
    df_base_filtrada
    .groupby('paciente_id')
    .apply(calcular_ingreso_egreso_ajustado)
    .reset_index()
)

print("=== INGRESO/EGRESO AJUSTADO ===")
print(f"Pacientes sin final clínico (flag):   {df_ingreso_egreso['flag_sin_final_clinico'].sum()}")
print(f"Pacientes con múltiples finales (flag): {df_ingreso_egreso['flag_multiples_finales'].sum()}")
print("\nDistribución tipo_egreso_final:")
print(df_ingreso_egreso['tipo_egreso_final'].value_counts())


In [ ]:
# ==============================================================================
# AJUSTE 3 — CONSTRUCCIÓN DE TRAYECTORIA CON NODO FINAL ORDENADO POR LÓGICA
# ==============================================================================

def construir_trayectoria_v1_ajustada(grp):
    """
    Construcción de trayectoria con prioridad lógica sobre fechas:
    
    1. Separa los traslados del df_traslados_strict (intermedios)
       y el episodio final (no traslado) desde df_base_filtrada.
    2. Ordena los traslados por fecha_egreso_origen.
    3. Ancla el episodio final al último lugar, independientemente
       de la fecha (prioridad lógica > orden temporal).
    4. Levanta flags en casos ambiguos.
    """
    grp = grp.sort_values('fecha_egreso_origen')

    flags = {
        'salto_inconsistente': 0,
        'loop': 0,
        'duplicado_consecutivo': 0
    }

    current_hospital = grp.iloc[0]['hospital_origen']
    hospitales = [current_hospital]

    for _, row in grp.iterrows():
        origen = row['hospital_origen']
        destino = row['hospital_destino']

        if origen != current_hospital:
            flags['salto_inconsistente'] = 1

        if destino == current_hospital:
            flags['duplicado_consecutivo'] = 1
        elif destino in hospitales:
            flags['loop'] = 1

        hospitales.append(destino)
        current_hospital = destino

    # Colapsar duplicados consecutivos
    hospitales_limpios = [hospitales[0]]
    for h in hospitales[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final_traslados': hospitales_limpios[-1],
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        **{f'flag_{k}': v for k, v in flags.items()}
    })


# Aplicar a pacientes con traslados
df_tray_ajustada_conectada = (
    df_traslados_filtrado
    .groupby('paciente_id')
    .apply(construir_trayectoria_v1_ajustada)
    .reset_index()
)

# Ahora — AJUSTE CLAVE: incorporar el nodo final desde df_base_filtrada
# El hospital del episodio final real (no-traslado) es el verdadero último nodo
df_nodo_final = (
    df_base_filtrada[df_base_filtrada['tipo_egreso'] != 'traslado']
    .sort_values(['paciente_id', 'fecha_egreso'])
    .groupby('paciente_id')
    .last()[['hospital_origen', 'tipo_egreso']]
    .rename(columns={'hospital_origen': 'hospital_nodo_final', 'tipo_egreso': 'tipo_egreso_final_ep'})
    .reset_index()
)

# Unir con trayectorias conectadas
df_tray_ajustada_conectada = df_tray_ajustada_conectada.merge(
    df_nodo_final, on='paciente_id', how='left'
)

# Flag: ¿el nodo final del traslado difiere del nodo final clínico?
df_tray_ajustada_conectada['flag_nodo_final_discrepante'] = (
    df_tray_ajustada_conectada['hospital_final_traslados'] !=
    df_tray_ajustada_conectada['hospital_nodo_final']
)

# Hospital final ajustado = hospital del episodio clínico final
# Si no existe (todos traslados), mantener el del último traslado
df_tray_ajustada_conectada['hospital_final'] = (
    df_tray_ajustada_conectada['hospital_nodo_final']
    .fillna(df_tray_ajustada_conectada['hospital_final_traslados'])
)

print("=== NODO FINAL DISCREPANTE (traslados vs clínico) ===")
print(df_tray_ajustada_conectada['flag_nodo_final_discrepante'].value_counts())


In [ ]:
# ==============================================================================
# TRAYECTORIAS TRIVIALES AJUSTADAS (sin traslados en df_traslados_strict)
# ==============================================================================

df_triviales_aj = df_base_filtrada[
    df_base_filtrada['paciente_id'].isin(pacientes_sin_traslado)
].copy()

def trayectoria_trivial_ajustada(grp):
    grp = grp.sort_values('fecha_ingreso')

    # Separar traslados de no-traslados
    intermedios = grp[grp['tipo_egreso'] == 'traslado']['hospital_origen'].tolist()
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        hospital_final = grp['hospital_origen'].iloc[-1]
        flag_sin_final = True
    else:
        # El hospital final es el del primer episodio no-traslado (más clínico)
        hospital_final = finales.sort_values('fecha_egreso').iloc[-1]['hospital_origen']
        flag_sin_final = False

    # Secuencia: intermedios + final
    h_seq = grp['hospital_origen'].tolist()
    hospitales_limpios = [h_seq[0]]
    for h in h_seq[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final': hospital_final,
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        'flag_salto_inconsistente': 0,
        'flag_loop': 0,
        'flag_duplicado_consecutivo': 0,
        'flag_nodo_final_discrepante': hospital_final != hospitales_limpios[-1],
        'flag_sin_final_clinico': flag_sin_final
    })

if len(df_triviales_aj) > 0:
    df_tray_triviales_aj = (
        df_triviales_aj
        .groupby('paciente_id')
        .apply(trayectoria_trivial_ajustada)
        .reset_index()
    )
else:
    df_tray_triviales_aj = pd.DataFrame()


In [ ]:
# ==============================================================================
# UNIFICACIÓN Y MERGE FINAL → df_pacientes_trayectorias_v1_ajustada
# ==============================================================================

df_tray_ajustada = pd.concat([
    df_tray_ajustada_conectada,
    df_tray_triviales_aj
], ignore_index=True)

# Merge con métricas ajustadas de ingreso/egreso
df_tray_ajustada_final = (
    df_tray_ajustada
    .merge(df_ingreso_egreso, on='paciente_id', how='left')
    .merge(
        df_base_filtrada.groupby('paciente_id')['hospital_origen']
        .count().reset_index().rename(columns={'hospital_origen': 'n_episodios'}),
        on='paciente_id', how='left'
    )
)

# Desenlace desde tipo_egreso_final (ya calculado con la lógica ajustada)
df_tray_ajustada_final['desenlace'] = df_tray_ajustada_final['tipo_egreso_final']

# Métricas de duración usando ingreso/egreso ajustado
df_tray_ajustada_final['duracion_total'] = (
    (df_tray_ajustada_final['fecha_egreso_paciente'] - df_tray_ajustada_final['fecha_ingreso_paciente'])
    .dt.days
)

# Guardar
df_tray_ajustada_final.to_excel("../data/final_data/df_pacientes_trayectorias_v1_ajustada.xlsx", index=False)

print("=== RESUMEN FINAL — V1 AJUSTADA ===")
print(f"Pacientes totales:                {len(df_tray_ajustada_final)}")
print(f"Con nodo final discrepante:       {df_tray_ajustada_final['flag_nodo_final_discrepante'].sum()}")
print(f"Sin final clínico (flag):         {df_tray_ajustada_final['flag_sin_final_clinico'].sum()}")
print(f"Con múltiples finales (flag):     {df_tray_ajustada_final.get('flag_multiples_finales', pd.Series(False)).sum()}")
print(f"\nDesenlace:")
print(df_tray_ajustada_final['desenlace'].value_counts())
print(f"\nHospitales únicos promedio:       {df_tray_ajustada_final['n_hospitales_unicos'].mean():.2f}")
print(f"Duración total promedio (días):   {df_tray_ajustada_final['duracion_total'].median():.1f} (mediana)")


In [ ]:
# ==============================================================================
# VALIDACIÓN — COMPARAR V1 original vs V1 AJUSTADA
# ==============================================================================

df_v1_orig = pd.read_excel("../data/final_data/df_pacientes_trayectorias_v1.xlsx")

comp = df_v1_orig[['paciente_id', 'hospital_final', 'desenlace']].merge(
    df_tray_ajustada_final[['paciente_id', 'hospital_final', 'desenlace', 'flag_nodo_final_discrepante']],
    on='paciente_id', suffixes=('_orig', '_aj'), how='inner'
)

comp['cambio_hospital_final'] = comp['hospital_final_orig'] != comp['hospital_final_aj']
comp['cambio_desenlace']      = comp['desenlace_orig'] != comp['desenlace_aj']

print("=== V1 ORIGINAL vs V1 AJUSTADA ===")
print(f"Pacientes comparados:              {len(comp)}")
print(f"Con cambio en hospital final:      {comp['cambio_hospital_final'].sum()} ({comp['cambio_hospital_final'].mean():.1%})")
print(f"Con cambio en desenlace:           {comp['cambio_desenlace'].sum()} ({comp['cambio_desenlace'].mean():.1%})")

# Muestra de casos que cambiaron
print("\n=== SAMPLE DE CASOS CON NODO FINAL DISTINTO ===")
sample_cambios = comp[comp['cambio_hospital_final']].sample(min(5, comp['cambio_hospital_final'].sum()), random_state=42)
for _, row in sample_cambios.iterrows():
    ep_cols = ['fecha_ingreso', 'fecha_egreso', 'hospital_origen', 'tipo_egreso']
    print(f"\nPaciente {row['paciente_id']}")
    print(f"  Hospital final ORIGINAL: {row['hospital_final_orig']}")
    print(f"  Hospital final AJUSTADO: {row['hospital_final_aj']}")
    from IPython.display import display
    display(df_base_filtrada[df_base_filtrada['paciente_id'] == row['paciente_id']][ep_cols])


---

# 🔧 METODOLOGÍA 1 — AJUSTADA (v1_ajustada)

Se incorporan dos correcciones al modelo v1 basadas en patrones observados en los datos:

**Problema 1 — Duplicados con traslados:**  
Muchos pacientes tienen varios episodios con `tipo_egreso == 'traslado'`
pero un solo episodio con `alta` o `muerte`. 
Ese episodio no-traslado es el **cierre clínico real**, 
pero a veces las fechas no lo ordenan correctamente.

**Problema 2 — Definición de ingreso/egreso:**  
- `fecha_ingreso_paciente = min(fecha_ingreso)` (siempre)
- `fecha_egreso_paciente = fecha_egreso` del episodio cuyo `tipo_egreso != 'traslado'`

**Estrategia:** prioridad lógica > fechas.


In [ ]:
# ==============================================================================
# AJUSTE 1 — CLASIFICACIÓN DE EPISODIOS POR ROL (intermedios vs final)
# ==============================================================================

# Marcar qué episodios son traslados (intermedios) vs finales clínicos
df_base_filtrada = df_base_filtrada.copy()

df_base_filtrada['es_traslado'] = df_base_filtrada['tipo_egreso'] == 'traslado'

# Por paciente: ¿tiene al menos un episodio NO traslado?
tiene_final = (
    df_base_filtrada
    .groupby('paciente_id')['es_traslado']
    .apply(lambda x: (x == False).any())
    .rename('tiene_final_clinico')
    .reset_index()
)

df_base_filtrada = df_base_filtrada.merge(tiene_final, on='paciente_id', how='left')

print("=== DISTRIBUCIÓN DE TIPOS DE EPISODIO ===")
print(df_base_filtrada['tipo_egreso'].value_counts())
print(f"\nPacientes CON episodio final clínico:    {tiene_final['tiene_final_clinico'].sum()}")
print(f"Pacientes SIN episodio final clínico:     {(~tiene_final['tiene_final_clinico']).sum()}")
print(f"  -> estos quedan flagueados, NO eliminados")


In [ ]:
# ==============================================================================
# AJUSTE 2 — FECHA DE INGRESO Y EGRESO REAL POR PACIENTE
# ==============================================================================

def calcular_ingreso_egreso_ajustado(grp):
    """
    Ingreso real  = min(fecha_ingreso)  [siempre, no depende del motivo]
    Egreso real   = fecha_egreso del episodio con tipo_egreso != 'traslado'
                    Si no existe, usar max(fecha_egreso) y marcar flag.
    """
    grp = grp.sort_values('fecha_ingreso')

    # Ingreso
    fecha_ingreso_paciente = grp['fecha_ingreso'].min()

    # Episodios finales (no traslado)
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        # Todos son traslado → inconsistente
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': grp['fecha_egreso'].max(),
            'flag_sin_final_clinico': True,
            'flag_multiples_finales': False,
            'tipo_egreso_final': 'desconocido'
        })
    elif len(finales) == 1:
        ep_final = finales.iloc[0]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': False,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })
    else:
        # Múltiples candidatos finales → usar el de mayor fecha y flaggear
        ep_final = finales.sort_values('fecha_egreso').iloc[-1]
        return pd.Series({
            'fecha_ingreso_paciente': fecha_ingreso_paciente,
            'fecha_egreso_paciente': ep_final['fecha_egreso'],
            'flag_sin_final_clinico': False,
            'flag_multiples_finales': True,
            'tipo_egreso_final': ep_final['tipo_egreso']
        })

df_ingreso_egreso = (
    df_base_filtrada
    .groupby('paciente_id')
    .apply(calcular_ingreso_egreso_ajustado)
    .reset_index()
)

print("=== INGRESO/EGRESO AJUSTADO ===")
print(f"Pacientes sin final clínico (flag):   {df_ingreso_egreso['flag_sin_final_clinico'].sum()}")
print(f"Pacientes con múltiples finales (flag): {df_ingreso_egreso['flag_multiples_finales'].sum()}")
print("\nDistribución tipo_egreso_final:")
print(df_ingreso_egreso['tipo_egreso_final'].value_counts())


In [ ]:
# ==============================================================================
# AJUSTE 3 — CONSTRUCCIÓN DE TRAYECTORIA CON NODO FINAL ORDENADO POR LÓGICA
# ==============================================================================

def construir_trayectoria_v1_ajustada(grp):
    """
    Construcción de trayectoria con prioridad lógica sobre fechas:
    
    1. Separa los traslados del df_traslados_strict (intermedios)
       y el episodio final (no traslado) desde df_base_filtrada.
    2. Ordena los traslados por fecha_egreso_origen.
    3. Ancla el episodio final al último lugar, independientemente
       de la fecha (prioridad lógica > orden temporal).
    4. Levanta flags en casos ambiguos.
    """
    grp = grp.sort_values('fecha_egreso_origen')

    flags = {
        'salto_inconsistente': 0,
        'loop': 0,
        'duplicado_consecutivo': 0
    }

    current_hospital = grp.iloc[0]['hospital_origen']
    hospitales = [current_hospital]

    for _, row in grp.iterrows():
        origen = row['hospital_origen']
        destino = row['hospital_destino']

        if origen != current_hospital:
            flags['salto_inconsistente'] = 1

        if destino == current_hospital:
            flags['duplicado_consecutivo'] = 1
        elif destino in hospitales:
            flags['loop'] = 1

        hospitales.append(destino)
        current_hospital = destino

    # Colapsar duplicados consecutivos
    hospitales_limpios = [hospitales[0]]
    for h in hospitales[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final_traslados': hospitales_limpios[-1],
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        **{f'flag_{k}': v for k, v in flags.items()}
    })


# Aplicar a pacientes con traslados
df_tray_ajustada_conectada = (
    df_traslados_filtrado
    .groupby('paciente_id')
    .apply(construir_trayectoria_v1_ajustada)
    .reset_index()
)

# Ahora — AJUSTE CLAVE: incorporar el nodo final desde df_base_filtrada
# El hospital del episodio final real (no-traslado) es el verdadero último nodo
df_nodo_final = (
    df_base_filtrada[df_base_filtrada['tipo_egreso'] != 'traslado']
    .sort_values(['paciente_id', 'fecha_egreso'])
    .groupby('paciente_id')
    .last()[['hospital_origen', 'tipo_egreso']]
    .rename(columns={'hospital_origen': 'hospital_nodo_final', 'tipo_egreso': 'tipo_egreso_final_ep'})
    .reset_index()
)

# Unir con trayectorias conectadas
df_tray_ajustada_conectada = df_tray_ajustada_conectada.merge(
    df_nodo_final, on='paciente_id', how='left'
)

# Flag: ¿el nodo final del traslado difiere del nodo final clínico?
df_tray_ajustada_conectada['flag_nodo_final_discrepante'] = (
    df_tray_ajustada_conectada['hospital_final_traslados'] !=
    df_tray_ajustada_conectada['hospital_nodo_final']
)

# Hospital final ajustado = hospital del episodio clínico final
# Si no existe (todos traslados), mantener el del último traslado
df_tray_ajustada_conectada['hospital_final'] = (
    df_tray_ajustada_conectada['hospital_nodo_final']
    .fillna(df_tray_ajustada_conectada['hospital_final_traslados'])
)

print("=== NODO FINAL DISCREPANTE (traslados vs clínico) ===")
print(df_tray_ajustada_conectada['flag_nodo_final_discrepante'].value_counts())


In [ ]:
# ==============================================================================
# TRAYECTORIAS TRIVIALES AJUSTADAS (sin traslados en df_traslados_strict)
# ==============================================================================

df_triviales_aj = df_base_filtrada[
    df_base_filtrada['paciente_id'].isin(pacientes_sin_traslado)
].copy()

def trayectoria_trivial_ajustada(grp):
    grp = grp.sort_values('fecha_ingreso')

    # Separar traslados de no-traslados
    intermedios = grp[grp['tipo_egreso'] == 'traslado']['hospital_origen'].tolist()
    finales = grp[grp['tipo_egreso'] != 'traslado']

    if len(finales) == 0:
        hospital_final = grp['hospital_origen'].iloc[-1]
        flag_sin_final = True
    else:
        # El hospital final es el del primer episodio no-traslado (más clínico)
        hospital_final = finales.sort_values('fecha_egreso').iloc[-1]['hospital_origen']
        flag_sin_final = False

    # Secuencia: intermedios + final
    h_seq = grp['hospital_origen'].tolist()
    hospitales_limpios = [h_seq[0]]
    for h in h_seq[1:]:
        if h != hospitales_limpios[-1]:
            hospitales_limpios.append(h)

    return pd.Series({
        'trayectoria_hospitalaria': str(hospitales_limpios),
        'hospital_inicio': hospitales_limpios[0],
        'hospital_final': hospital_final,
        'n_hospitales_unicos': len(set(hospitales_limpios)),
        'flag_salto_inconsistente': 0,
        'flag_loop': 0,
        'flag_duplicado_consecutivo': 0,
        'flag_nodo_final_discrepante': hospital_final != hospitales_limpios[-1],
        # flag_sin_final_clinico viene de df_ingreso_egreso via merge posterior
    })

if len(df_triviales_aj) > 0:
    df_tray_triviales_aj = (
        df_triviales_aj
        .groupby('paciente_id')
        .apply(trayectoria_trivial_ajustada)
        .reset_index()
    )
else:
    df_tray_triviales_aj = pd.DataFrame()


In [ ]:
# ==============================================================================
# UNIFICACIÓN Y MERGE FINAL → df_pacientes_trayectorias_v1_ajustada
# ==============================================================================

df_tray_ajustada = pd.concat([
    df_tray_ajustada_conectada,
    df_tray_triviales_aj
], ignore_index=True)

# Merge con métricas ajustadas de ingreso/egreso
df_tray_ajustada_final = (
    df_tray_ajustada
    .merge(df_ingreso_egreso, on='paciente_id', how='left')
    .merge(
        df_base_filtrada.groupby('paciente_id')['hospital_origen']
        .count().reset_index().rename(columns={'hospital_origen': 'n_episodios'}),
        on='paciente_id', how='left'
    )
)

# Desenlace desde tipo_egreso_final (ya calculado con la lógica ajustada)
df_tray_ajustada_final['desenlace'] = df_tray_ajustada_final['tipo_egreso_final']

# Métricas de duración usando ingreso/egreso ajustado
df_tray_ajustada_final['duracion_total'] = (
    (df_tray_ajustada_final['fecha_egreso_paciente'] - df_tray_ajustada_final['fecha_ingreso_paciente'])
    .dt.days
)

# Guardar
df_tray_ajustada_final.to_excel("../data/final_data/df_pacientes_trayectorias_v1_ajustada.xlsx", index=False)

print("=== RESUMEN FINAL — V1 AJUSTADA ===")
print(f"Pacientes totales:                {len(df_tray_ajustada_final)}")
print(f"Con nodo final discrepante:       {df_tray_ajustada_final['flag_nodo_final_discrepante'].sum()}")
print(f"Sin final clinico (flag):         {df_tray_ajustada_final['flag_sin_final_clinico'].sum()}")
print(f"Con multiples finales (flag):     {df_tray_ajustada_final['flag_multiples_finales'].sum()}")
print(f"\nDesenlace:")
print(df_tray_ajustada_final['desenlace'].value_counts())
print(f"\nHospitales unicos promedio:       {df_tray_ajustada_final['n_hospitales_unicos'].mean():.2f}")
print(f"Duracion total mediana (dias):   {df_tray_ajustada_final['duracion_total'].median():.1f}")


In [ ]:
# ==============================================================================
# VALIDACIÓN — COMPARAR V1 original vs V1 AJUSTADA
# ==============================================================================

df_v1_orig = pd.read_excel("../data/final_data/df_pacientes_trayectorias_v1.xlsx")

comp = df_v1_orig[['paciente_id', 'hospital_final', 'desenlace']].merge(
    df_tray_ajustada_final[['paciente_id', 'hospital_final', 'desenlace', 'flag_nodo_final_discrepante']],
    on='paciente_id', suffixes=('_orig', '_aj'), how='inner'
)

comp['cambio_hospital_final'] = comp['hospital_final_orig'] != comp['hospital_final_aj']
comp['cambio_desenlace']      = comp['desenlace_orig'] != comp['desenlace_aj']

print("=== V1 ORIGINAL vs V1 AJUSTADA ===")
print(f"Pacientes comparados:              {len(comp)}")
print(f"Con cambio en hospital final:      {comp['cambio_hospital_final'].sum()} ({comp['cambio_hospital_final'].mean():.1%})")
print(f"Con cambio en desenlace:           {comp['cambio_desenlace'].sum()} ({comp['cambio_desenlace'].mean():.1%})")

# Muestra de casos que cambiaron
print("\n=== SAMPLE DE CASOS CON NODO FINAL DISTINTO ===")
sample_cambios = comp[comp['cambio_hospital_final']].sample(min(5, comp['cambio_hospital_final'].sum()), random_state=42)
for _, row in sample_cambios.iterrows():
    ep_cols = ['fecha_ingreso', 'fecha_egreso', 'hospital_origen', 'tipo_egreso']
    print(f"\nPaciente {row['paciente_id']}")
    print(f"  Hospital final ORIGINAL: {row['hospital_final_orig']}")
    print(f"  Hospital final AJUSTADO: {row['hospital_final_aj']}")
    from IPython.display import display
    display(df_base_filtrada[df_base_filtrada['paciente_id'] == row['paciente_id']][ep_cols])
